# Step 5 - Sentiment model + factor analysis

Second axis: predict how the audience will react (negative / neutral / positive) from pre-post
content features, and find the factors / words associated with positive vs negative reception.
Same leakage rule: no outcome signals (views/likes/comment counts) as inputs.

In [ ]:
# load features + pick sentiment target
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")

num_features = [
    "title_len_words", "desc_len_words", "trans_len_words",
    "title_sentiment", "title_has_question", "title_upper_ratio",
    "kw_price", "kw_range", "kw_charging",
    "pub_hour", "pub_dow", "pub_month",
    "duration_min", "channel_freq", "has_description", "has_transcript",
]
X = df[num_features]
y = df["sentiment_class"]                 # 0=negative, 1=neutral, 2=positive
print("X:", X.shape, "| classes:", y.value_counts().to_dict())

In [ ]:
# baseline + meta-only RandomForest (cross-validated)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy = DummyClassifier(strategy="most_frequent")
print("Baseline macro-F1:", round(f1_score(y, cross_val_predict(dummy, X, y, cv=cv), average="macro"), 3))

rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=42, n_jobs=-1)
pred = cross_val_predict(rf, X, y, cv=cv)
print("Meta-only RF macro-F1:", round(f1_score(y, pred, average="macro"), 3))
print("\n", classification_report(y, pred, target_names=["negative", "neutral", "positive"]))
print("Confusion (rows=true):\n", confusion_matrix(y, pred))

In [ ]:
# meta factor importance (permutation on held-out set)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
rf.fit(Xtr, ytr)
perm = permutation_importance(rf, Xte, yte, n_repeats=20, random_state=42, scoring="f1_macro")
imp = (pd.DataFrame({"feature": num_features, "importance": perm.importances_mean})
       .sort_values("importance", ascending=False))
print("=== Factors driving SENTIMENT (high to low) ===")
print(imp.to_string(index=False))

In [ ]:
# TF-IDF text-only model (best approach on the virality axis) + word insight
text_clf = Pipeline([
    ("tf", TfidfVectorizer(max_features=2000, min_df=5, ngram_range=(1, 2),
                           stop_words="english", sublinear_tf=True)),
    ("lr", LogisticRegression(max_iter=2000)),
])
tpred = cross_val_predict(text_clf, df["text_all"], y, cv=cv)
print("Text-only (logreg) macro-F1:", round(f1_score(y, tpred, average="macro"), 3))
print(classification_report(y, tpred, target_names=["negative", "neutral", "positive"]))

# which words are associated with positive vs negative reception (descriptive)
tfidf = TfidfVectorizer(max_features=3000, min_df=5, ngram_range=(1, 2),
                        stop_words="english", sublinear_tf=True)
Xt = tfidf.fit_transform(df["text_all"])
clf = LogisticRegression(max_iter=2000, C=1.0).fit(Xt, y)
terms = np.array(tfidf.get_feature_names_out())
for cls, name in [(2, "POSITIVE reception"), (0, "NEGATIVE reception")]:
    idx = list(clf.classes_).index(cls)
    top = np.argsort(clf.coef_[idx])[-15:][::-1]
    print(f"\nTop tokens -> {name}:")
    print(", ".join(terms[top]))